# 9 WorkFlow Analista Senior

### 9.5 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
enriquecer
<br>El Analista Sr corre sus scripts en máquinas virtuales de al menos 128 GB de RAM en Toronto, creando una virtual machine para cada corrida.
<br>Estas virtual machines se auto-suicidarán a los 30 minutos de haber terminado de procesar.

## 9.7  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 02:03:45 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671183,35.9,1479641,79.1,1479641,79.1
Vcells,1242599,9.5,8388608,64.0,1978699,15.1


In [3]:
# creo PARAM
PARAM <- list()
PARAM$tiempos <- list()
PARAM$tiempos$inicio_notebook <- format(Sys.time(), "%a %b %d %X %Y")

In [4]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

In [5]:
PARAM$semilla_primigenia <- 761927

PARAM$experimento <- 761927
PARAM$dataset <- "analistasr_competencia_2026.csv.gz"

#### Parametros


#### Carpeta del Experimento

In [6]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF_PRED_DT_", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.7.1   Preprocesamiento del dataset (se lee archivo ya pre-procesado)

### 9.7.2 Modelado

In [7]:
#Leo el archivo ya procesdo (con FE)
dataset <- fread("/content/buckets/b1/datasets/dataset_procesado_exp_colab.csv.gz")

#### 9.7.2.1 Training Strategy


In [8]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103
)

PARAM$trainingstrategy$training_pct <- 0.2


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [9]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [10]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [11]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

Loading required package: lightgbm



In [12]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 163906

####  9.7.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Cantidad de iteraciones inteligentes de la Optimizacion Bayesiana = **50**


In [13]:
# paquetes necesarios para la Bayesian Optimization
if(!require("DiceKriging")) install.packages("DiceKriging")
require("DiceKriging")

if(!require("mlrMBO")) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers


Attaching package: ‘ParamHelpers’


The following object is masked from ‘package:R.utils’:

    isVector



Attaching package: ‘mlr’


The following objects are masked from ‘package:R.utils’:

    resample, setThreshold


Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘checkmate’


The following object is masked from ‘package:DiceKriging’:

    checkNames


The following object is masked from ‘package:R.utils’:

    asInt



Attaching package: ‘smoof’


The following objects are masked from ‘package:R.oo’:

    getDescription, getName




**Definición de la Bayesian Optimization**
<br> Si se desea optimizar un hiperparámetro que esta como fijo, debe QUITARSE de param_fijos y agregarse a PARAM$hipeparametertuning$hs

In [14]:
# 40 es el minimo razonable,  un buen analista Sr aumentara este valor
PARAM$hipeparametertuning$num_interations <- 50

# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  extra_trees= FALSE,

  #num_leaves= 128,
  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0.0, # min_gain_to_split >= 0.0
  min_data_in_leaf= 0,
  #min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  boosting_type= "dart",

  max_drop= 50, # <=0 means no limit

  max_bin= 31,
  early_stopping_rounds= 0

)


In [15]:
PARAM$hipeparametertuning$hs <- makeParamSet(
  makeNumericParam("num_iterations", lower= 10.0, upper= 12.0, trafo= function(x) round(2^x) ),   # 1024 a 4096
  makeNumericParam("num_leaves", lower= 4.0, upper= 9.0, trafo= function(x) round(2^x) ),
  makeNumericParam("learning_rate", lower= -7.0, upper= -4.5, trafo= function(x) 2^x ),
  makeNumericParam("feature_fraction", lower= 0.15, upper= 0.70),
  makeNumericParam("min_sum_hessian_in_leaf", lower= -2.0, upper= 8.0, trafo= function(x) 2^x),
  makeNumericParam("drop_rate", lower= -10.0, upper= -4.0, trafo= function(x) 2^x),   # 0.001 a 0.06
  makeNumericParam("skip_drop", lower= 0.0, upper= 0.90)
)

Función "señora caja negra"  que es llamada para verificar la realidad por la Bayesian Optimization

In [16]:
# En  x llegan los parmaetros de la bayesiana
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

EstimarGanancia_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[x$num_iterations]]


  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return(AUC)
}

seteo de la Bayesian Optimization (complejo)
<br> copiado y pegado de la documentación de la librería

In [17]:
configureMlr(show.learner.output = FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
    fn= EstimarGanancia_AUC_lightgbm, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando AUC
    noisy= FALSE,
    par.set= PARAM$hipeparametertuning$hs,
    has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
    save.on.disk.at.time= 600,
    save.file.path= "HT.RDATA"
)

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
    ctrl,
    iters= PARAM$hipeparametertuning$num_interations  # cantidad de iteraciones inteligentes
)

# defino el método estandar para la creacion de los puntos iniciales
#   los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit = makeMBOInfillCritEI())

# mas configuraciones
surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace = TRUE)
)

Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer

Si corrío a medias y llegó a las iteraciones inteligentes, en el archivo binario HT.RDATA quedó lo ya procesado y es utilizado para retomar la corrida desde lo último que llegó a grabar.

In [18]:
#Inicio Optimización Bayesiana
PARAM$tiempos$inicio_OB <- format(Sys.time(), "%a %b %d %X %Y")
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 02:09:28 2026"

In [19]:
# inicio la optimizacion bayesiana

if (!file.exists("HT.RDATA")) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue("HT.RDATA") # retomo en caso que ya exista
}

Warning message in mboContinue("HT.RDATA"):
“Tuning ended with term.iter. No need to continue. Simply returning stored result.”


la bayesian optimization ha corrido, extraigo los mejores hiperparametros

In [20]:
#Fin Optimización 
PARAM$tiempos$fin_OB <- format(Sys.time(), "%a %b %d %X %Y")
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 02:09:28 2026"

In [21]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y, -num_iterations)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file="BO_log.txt",
  sep="\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]

print(PARAM$out$lgbm$mejores_hiperparametros)

   num_iterations num_leaves learning_rate feature_fraction
            <num>      <num>         <num>            <num>
1:       11.99672    6.54871      -4.93248        0.3282783
   min_sum_hessian_in_leaf drop_rate skip_drop
                     <num>     <num>     <num>
1:               0.1671135 -8.911675 0.3395526


### 9.7.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [22]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

PARAM$trainingstrategy$final_train_nrows <-nrow( dfinal_train)
nrow( dfinal_train) # verifico el tamaño

[1] 4218774

##### Final Training Hyperparameters

In [23]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- copy(fijos)

x <- PARAM$out$lgbm$mejores_hiperparametros
param_final$num_iterations <- round( 2^x$num_iterations )
param_final$num_leaves <- round( 2^x$num_leaves )
param_final$learning_rate <- 2^x$learning_rate
param_final$feature_fraction <- x$feature_fraction
param_final$min_sum_hessian_in_leaf <- 2^x$min_sum_hessian_in_leaf
param_final$drop_rate <- 2^x$drop_rate
param_final$skip_drop <- x$skip_drop




In [24]:
param_final

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] TRUE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$verbosity
[1] -100

$force_row_wise
[1] TRUE

$seed
[1] 761927

$extra_trees
[1] FALSE

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_data_in_leaf
[1] 0

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$boosting_type
[1] "dart"

$max_drop
[1] 50

$max_bin
[1] 31

$num_iterations
[1] 4087

$num_leaves
[1] 94

$learning_rate
[1] 0.03274731

$feature_fraction
[1] 0.3282783

$min_sum_hessian_in_leaf
[1] 1.12281

$drop_rate
[1] 0.002076436

$skip_drop
[1] 0.3395526

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [25]:
# poner este valor en 5 o en 1  si se está muy ajustado de tiempo

PARAM$FT$semillerio <- 10  # cantidad de semillas

In [26]:
if(!require("primes")) install.packages("primes")
require("primes")

Loading required package: primes



In [27]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# me quedo con PARAM$semillerio  primos al azar
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

cat( PARAM$FT$semillas)

260959 779081 376417 332147 825203 879863 982621 985997 926129 180647

In [28]:
primero <- TRUE

crear_modelo_final <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if( !file.exists(nombre_arch) )
  {
    param_final$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )

    lgb.save(final_model, nombre_arch) # grabo el modelo"

    # grabo la primer importancia de variables
    #  Natalia : da lo mismo cual se guarda
    if( primero)
    {
      primero <<- FALSE
      tb_importancia <- as.data.table(lgb.importance(final_model))
      archivo_importancia <- "impo.txt"

      fwrite( tb_importancia,
        file= archivo_importancia,
        sep= "\t"
      )
    }

    rm(final_model)
    gc(full = TRUE, verbose=FALSE)
  }
}

In [29]:
# Inicio 10 corridas modelo final
PARAM$tiempos$inicio_ModFinal <- format(Sys.time(), "%a %b %d %X %Y")
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 02:13:28 2026"

In [30]:
gc(full = TRUE, verbose=FALSE)
dir.create("modelos", showWarnings =FALSE)

primero <- TRUE
#for( sem in PARAM$FT$semillas)  crear_modelo_final(sem)

PARAM$tiempos$modelo_final_ciclofor <- list()   # una sola vez, antes del ciclo

for( sem in PARAM$FT$semillas)
{
    crear_modelo_final(sem)

    hora <- format(Sys.time(), "%a %b %d %X %Y")

    # agrega un registro nuevo al final de la lista
    PARAM$tiempos$modelo_final_ciclofor[[ length(PARAM$tiempos$modelo_final_ciclofor) + 1 ]] <-
        list(semilla = sem, fin = hora)

    cat("semilla:", sem, "- fin:", hora, "\n")
}

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2669057,142.6,5001587,267.2,3758552,200.8
Vcells,6280664058,47917.7,9553734982,72889.3,8965122748,68398.5


Warning message in gsum(internal_count):
“The sum of an integer column for a group was more than type 'integer' can hold so the result has been coerced to 'numeric' automatically for convenience.”


semilla: 260959 - fin: Tue Nov 03 03:43:53 2026 
semilla: 779081 - fin: Tue Nov 03 05:15:05 2026 
semilla: 376417 - fin: Tue Nov 03 06:49:12 2026 
semilla: 332147 - fin: Tue Nov 03 08:22:09 2026 
semilla: 825203 - fin: Tue Nov 03 09:55:22 2026 
semilla: 879863 - fin: Tue Nov 03 11:27:26 2026 
semilla: 982621 - fin: Tue Nov 03 12:58:47 2026 
semilla: 985997 - fin: Tue Nov 03 14:32:03 2026 
semilla: 926129 - fin: Tue Nov 03 16:05:35 2026 
semilla: 180647 - fin: Tue Nov 03 17:38:58 2026 


#### Scoring

Aplico el modelo final a los datos del futuro

In [31]:
#Inicio Prediccion sobre datos del futuro
PARAM$tiempos$inicio_prediccion<- format(Sys.time(), "%a %b %d %X %Y")
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 17:38:58 2026"

In [32]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [33]:
# aplico final_model   a dfuture

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := 0]

datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])

In [34]:
agregar_a_ensemble <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  final_model <- lgb.load(nombre_arch)

  prediccion <- predict(
    final_model,
    datos_matrix
  )

  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]

  rm(final_model)
  rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
}

In [35]:
for( isem in seq(length(PARAM$FT$semillas)) )
{
  sem <- PARAM$FT$semillas[ isem ]
  agregar_a_ensemble( sem )
  gc(full = TRUE, verbose=FALSE)
}

rm( datos_matrix)
gc(full = TRUE, verbose=FALSE)

tb_prediccion[, prob := prob /length(PARAM$FT$semillas) ]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2714482,145.0,8760976,467.9,21389098,1142.4
Vcells,4340637456,33116.5,9553734982,72889.3,8965122748,68398.5


In [36]:
# veo que hay en tb_prediccion
tb_prediccion

numero_de_cliente,prob,prob_1,prob_2,prob_3,prob_4,prob_5,prob_6,prob_7,prob_8,prob_9,prob_10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
11673459,0.0006437291,5.946872e-04,0.0005744437,4.769676e-04,0.0005910449,0.0005280900,6.156152e-04,0.0007013995,0.0008786683,8.998027e-04,0.0005765721
11673463,0.0011379512,1.176051e-03,0.0012068533,1.612726e-03,0.0009549303,0.0010157378,9.448993e-04,0.0011124553,0.0010265974,1.154564e-03,0.0011746982
11674064,0.0028867081,2.602929e-03,0.0023797415,3.239732e-03,0.0035229484,0.0024896690,3.011119e-03,0.0023301194,0.0033517615,3.397700e-03,0.0025413613
11674166,0.0003257996,4.613956e-04,0.0002570119,2.933103e-04,0.0002948733,0.0003370172,3.687200e-04,0.0003071892,0.0002854053,3.369487e-04,0.0003161244
11674316,0.0088763159,9.260270e-03,0.0078610978,9.827078e-03,0.0086502749,0.0093333436,9.026739e-03,0.0091242759,0.0081609960,7.499853e-03,0.0100192310
11674337,0.0006126759,5.730035e-04,0.0006689865,6.815468e-04,0.0004702665,0.0004880293,6.034943e-04,0.0008357966,0.0007145190,4.994465e-04,0.0005916697
11674528,0.0113674475,1.138452e-02,0.0105742622,1.281768e-02,0.0124142613,0.0098393358,1.090211e-02,0.0110007071,0.0118624572,1.150211e-02,0.0113770398
11674689,0.0011731359,9.107137e-04,0.0012025420,1.006371e-03,0.0012398409,0.0014703662,1.089860e-03,0.0013169696,0.0008298244,1.295204e-03,0.0013696677
11674959,0.0375746574,2.586733e-02,0.0255182442,5.391036e-02,0.0475040070,0.0303167418,4.228405e-02,0.0374155149,0.0362962922,3.966867e-02,0.0369653586


In [37]:
# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

In [38]:
#Fin Prediccion sobre datos del futuro
PARAM$tiempos$fin_predicción <- format(Sys.time(), "%a %b %d %X %Y")
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 17:41:29 2026"

#### Curva de Ganancia

Genero las salidas

In [39]:
# asigno clase ternaria
tb_prediccion[, clase_ternaria := dfuture$clase_ternaria ]

# asigno ganancias
tb_prediccion[, ganancia := -0.025 ]
tb_prediccion[clase_ternaria=="BAJA+2", ganancia := 0.975 ]

# reordeno, y acumulada
setorder( tb_prediccion, -prob )
tb_prediccion[, gan_acum := cumsum(ganancia)]

# media movil de ancho 400
tb_prediccion[,
  gan_suavizada := frollmean(
    x= gan_acum,
    n= 400,
    align= "center",
    na.rm= TRUE,
    hasNA= TRUE
  )
]

Warning message in froll(fun = "mean", x = x, n = n, fill = fill, algo = algo, align = align, :
“hasNA is deprecated, use has.nf instead”


In [40]:
tb_prediccion[, sum(!is.na(gan_suavizada) )]
tb_prediccion[, .N]

[1] 164080

[1] 164479

In [41]:
# Este es la ganancia a reportar
resultado <- list()
resultado$ganancia_suavizada_max <- max( tb_prediccion$gan_suavizada, na.rm=TRUE )
options(digits= 8)
resultado$envios <- which.max( tb_prediccion$gan_suavizada)
PARAM$resultado <- resultado
resultado

$ganancia_suavizada_max
[1] 598.245

$envios
[1] 13613

In [42]:
# grabo las ganancias
fwrite( tb_prediccion,
  file= "ganancias.txt",
  sep= "\t"
)

In [43]:
# genero el grafico (espacio variable)

tb_prediccion[, envios:= .I]

pdf("curva_de_ganancia_ejesdinamicos.pdf")

x_lim <- c(0, max(tb_prediccion$envios, na.rm= TRUE))
y_lim <- range(0, tb_prediccion$gan_acum, na.rm= TRUE)

plot(
  x= tb_prediccion$envios,
  y= tb_prediccion$gan_acum,
  type= "l",
  col= "gray",
  xlim= x_lim,
  ylim= y_lim,
  main= paste0("Mejor gan prom= ", as.integer(resultado$ganancia_suavizada_max), " envios= ", resultado$envios),
  xlab= "Envios",
  ylab= "Ganancia",
  panel.first= grid()
)

dev.off()


agg_record_452672ab372cd 
                       2

In [44]:
#Finaliza Notebook
PARAM$tiempos$fin_notebook <- format(Sys.time(), "%a %b %d %X %Y")

In [45]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [46]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 03 17:41:30 2026"